# Estequiometria: o mol na reação balanceada
### Coeficientes como proporção em mol e reagente limitante

**Objetivo:** Usar os coeficientes da equação balanceada como proporção em mols para calcular reagentes e produtos, identificando o reagente limitante.

### Como usar este notebook

Execute as células na ordem, de cima para baixo. As células Markdown apresentam os conceitos; as células de código fazem as contas e geram as visualizações.

> **Moléculas em 3D:** clique na figura e **arraste o mouse** — ela gira em 360° em qualquer direção. Use a roda do mouse (ou dois dedos no celular) para dar zoom. As cores seguem a convenção dos químicos (C = cinza, H = branco, O = vermelho, N = azul).

In [ ]:
# @title Preparar o ambiente (só precisa rodar uma vez)
import importlib.util, subprocess, sys

def _tem(modulo):
    return importlib.util.find_spec(modulo) is not None

for pacote in ("rdkit", "py3Dmol"):
    if not _tem(pacote):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=True)

import rdkit, py3Dmol
print("rdkit", rdkit.__version__, "| py3Dmol", py3Dmol.__version__)

## 1. Conceito principal

Na equação balanceada, os **coeficientes** são proporções em **mols** — não apenas "números bonitos". Para a combustão do metano:

$$\mathrm{CH_4 + 2\,O_2 \longrightarrow CO_2 + 2\,H_2O}$$

a leitura em mols é: **1 mol de $\mathrm{CH_4}$ reage com 2 mols de $\mathrm{O_2}$ e produz 1 mol de $\mathrm{CO_2}$ e 2 mols de $\mathrm{H_2O}$**.

A regra geral:

$$\boxed{n_{\text{produto}} = n_{\text{reagente}} \cdot \frac{\text{coef.}_{\text{produto}}}{\text{coef.}_{\text{reagente}}}}$$

O **reagente limitante** é o que acaba primeiro. Para descobrir, divida os mols disponíveis pelo coeficiente de cada reagente — o menor resultado é o limitante:

$$\boxed{\text{limitante} = \min\left(\frac{n_i}{\text{coef.}_i}\right)}$$

## 2. Exemplo resolvido passo a passo

**Pergunta:** com $0{,}400\ \mathrm{mol}$ de $\mathrm{CH_4}$ (e oxigênio em excesso), quantos mols e gramas de $\mathrm{O_2}$ são consumidos e de $\mathrm{CO_2}$ e $\mathrm{H_2O}$ são produzidos?

**Passo 1 — oxigênio consumido.**

$$n(\mathrm{O_2}) = 0{,}400 \cdot \frac{2}{1} = \boxed{0{,}800\ \mathrm{mol}} \;\Rightarrow\; m = 0{,}800\cdot 32{,}00 = 25{,}6\ \mathrm{g}$$

**Passo 2 — produtos formados.**

$$n(\mathrm{CO_2}) = 0{,}400\cdot\frac{1}{1} = \boxed{0{,}400\ \mathrm{mol}} \;\Rightarrow\; 0{,}400\cdot 44{,}01 = 17{,}6\ \mathrm{g}$$

$$n(\mathrm{H_2O}) = 0{,}400\cdot\frac{2}{1} = \boxed{0{,}800\ \mathrm{mol}} \;\Rightarrow\; 0{,}800\cdot 18{,}02 = 14{,}4\ \mathrm{g}$$

**Conferência de massa:** reagentes $= 0{,}400\cdot 16{,}04 + 25{,}6 = 32{,}0\ \mathrm{g}$; produtos $= 17{,}6 + 14{,}4 = 32{,}0\ \mathrm{g}$. A massa se conserva — como manda a lei de Lavoisier.

**Reagente limitante:** se tivéssemos $0{,}500\ \mathrm{mol}$ de $\mathrm{CH_4}$ e $0{,}800\ \mathrm{mol}$ de $\mathrm{O_2}$, teríamos $0{,}500/1 = 0{,}500$ e $0{,}800/2 = 0{,}400$. O menor é o $\mathrm{O_2}$: ele é o limitante e sobram $0{,}100\ \mathrm{mol}$ de $\mathrm{CH_4}$.

## 3. Conferindo o cálculo com Python

A próxima célula aplica a regra dos coeficientes e verifica a conservação de massa.

In [ ]:
# @title Contas do exemplo
from rdkit import Chem
from rdkit.Chem import Descriptors

def massa_molar(smiles):
    return Descriptors.MolWt(Chem.MolFromSmiles(smiles))

M_CH4 = massa_molar("C")
M_O2 = massa_molar("O=O")
M_CO2 = massa_molar("O=C=O")
M_H2O = massa_molar("O")

n_ch4 = 0.400
n_o2 = n_ch4 * 2 / 1
n_co2 = n_ch4 * 1 / 1
n_h2o = n_ch4 * 2 / 1

print(f"CH4:  {n_ch4:.3f} mol ({n_ch4 * M_CH4:.2f} g)")
print(f"O2:   {n_o2:.3f} mol ({n_o2 * M_O2:.2f} g)")
print(f"CO2:  {n_co2:.3f} mol ({n_co2 * M_CO2:.2f} g)")
print(f"H2O:  {n_h2o:.3f} mol ({n_h2o * M_H2O:.2f} g)")

m_reag = n_ch4 * M_CH4 + n_o2 * M_O2
m_prod = n_co2 * M_CO2 + n_h2o * M_H2O
print(f"\nMassa reagentes: {m_reag:.2f} g | produtos: {m_prod:.2f} g")

# Reagente limitante: 0,500 mol CH4 + 0,800 mol O2
print(f"\nCH4: 0,500/1 = {0.500/1:.3f} | O2: 0,800/2 = {0.800/2:.3f} -> O2 é o limitante")

## 4. Visualização

### 4.1 A molécula de metano em 3D — arraste para girar

A figura abaixo é interativa. Arraste para girar em 360°, use a roda do mouse para aproximar. O metano é o combustível da reação: o carbono (cinza) fica no centro e os quatro hidrogênios (brancos) apontam para os vértices de um tetraedro.

In [ ]:
# @title Metano em 3D interativo
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol

def molecula_3d(smiles, largura=640, altura=420):
    """SMILES -> visualização 3D interativa (arrastar = girar, rolar = zoom)."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))   # H explícitos: aparecem na figura
    parametros = AllChem.ETKDGv3()
    parametros.randomSeed = 0xC0FFEE               # mesmo desenho em toda execução
    if AllChem.EmbedMolecule(mol, parametros) != 0:
        raise RuntimeError("não foi possível gerar a geometria 3D")
    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMolecule(mol)          # relaxa as distâncias de ligação
    else:
        AllChem.UFFOptimizeMolecule(mol)

    visor = py3Dmol.view(width=largura, height=altura)
    visor.addModel(Chem.MolToMolBlock(mol), "mol")
    visor.setStyle({"stick": {"radius": 0.13, "colorscheme": "Jmol"},
                    "sphere": {"scale": 0.28, "colorscheme": "Jmol"}})
    visor.setBackgroundColor("#ffffff")
    visor.zoomTo()
    return visor

molecula_3d("C")   # metano: CH4 (combustível da reação)

### 4.2 Reagentes e produtos em 2D, com os risquinhos das ligações

Equação: $\mathrm{CH_4 + 2\,O_2 \rightarrow CO_2 + 2\,H_2O}$. No desenho 2D cada traço é uma ligação: **um** traço = simples, **dois** = dupla. Repare que o $\mathrm{O_2}$ tem ligação dupla (O=O) e o $\mathrm{CO_2}$ tem duas ligações duplas.

> **Detalhe técnico que vale a pena saber:** o desenho do RDKit corrompe acentos e subscritos na legenda. Por isso as legendas dentro das figuras do RDKit ficam em ASCII; o texto explicativo com acento vai no markdown.

In [ ]:
# @title Galeria 2D
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import SVG

nomes = ["metano (CH4)", "oxigenio (O2)", "gas carbonico (CO2)", "agua (H2O)"]
smiles = ["C", "O=O", "O=C=O", "O"]
imagem = Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in smiles],
                              molsPerRow=2, subImgSize=(320, 260), legends=nomes, useSVG=True)
# dentro do notebook o RDKit já devolve um objeto SVG; fora dele, devolve texto
imagem if hasattr(imagem, "data") else SVG(imagem)

### 4.3 Mols antes e depois da reação

In [ ]:
# @title Mols antes e depois
import numpy as np
import matplotlib.pyplot as plt

especies = ["CH₄", "O₂", "CO₂", "H₂O"]
antes = [0.400, 0.800, 0.000, 0.000]
depois = [0.000, 0.000, 0.400, 0.800]

x = np.arange(len(especies))
largura = 0.35
plt.figure(figsize=(7.5, 4.5))
plt.bar(x - largura / 2, antes, largura, label="antes (reagentes)")
plt.bar(x + largura / 2, depois, largura, label="depois (produtos)")
plt.xticks(x, especies)
plt.ylabel("Quantidade de matéria (mol)")
plt.title("Combustão do metano: os mols se transformam")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Como interpretar o resultado

- Os coeficientes são **proporções em mol**: 1 mol de $\mathrm{CH_4}$ pede 2 mols de $\mathrm{O_2}$ — não "1 molécula pede 2 moléculas" (embora a proporção seja a mesma, a unidade de cálculo é o mol).
- O gráfico mostra a transformação: os mols de reagentes viram mols de produtos. A massa total se conserva ($32{,}0\ \mathrm{g}$ de cada lado).
- No 3D, o metano é tetraédrico: gire a figura e conte os quatro hidrogênios ao redor do carbono.
- Para achar o limitante, compare $n/\text{coef.}$ de cada reagente — o menor manda na reação inteira.

## 6. Exercício de fixação

Na síntese da amônia, $\mathrm{N_2 + 3\,H_2 \rightarrow 2\,NH_3}$:

**a)** Com $2{,}00\ \mathrm{mol}$ de $\mathrm{H_2}$ (e $\mathrm{N_2}$ em excesso), quantos mols e gramas de $\mathrm{NH_3}$ são formados? ($M = 17{,}03\ \mathrm{g/mol}$.)

**b)** Se houver $1{,}00\ \mathrm{mol}$ de $\mathrm{N_2}$ e $2{,}00\ \mathrm{mol}$ de $\mathrm{H_2}$, qual é o reagente limitante?

Tente prever os resultados antes de executar a célula seguinte.

In [ ]:
# @title Exercício
# a) N2 + 3 H2 -> 2 NH3, com 2,00 mol de H2
n_nh3 = 2.00 * 2 / 3
m_nh3 = n_nh3 * 17.03
print(f"a) n(NH3) = {n_nh3:.2f} mol -> m = {m_nh3:.1f} g")

# b) limitante: N2: 1,00/1 = 1,00 | H2: 2,00/3 = 0,67 -> H2 é o limitante
print(f"b) N2: 1,00/1 = {1.00/1:.2f} | H2: 2,00/3 = {2.00/3:.2f} -> H2 é o limitante")

## 7. Resumo

- Coeficientes = proporção em mol: $n_{\text{prod}} = n_{\text{reag}} \cdot (\text{coef.}_{\text{prod}}/\text{coef.}_{\text{reag}})$.
- Reagente limitante: menor $n/\text{coef.}$ entre os reagentes.
- A massa se conserva: massa dos reagentes = massa dos produtos.
- Visualização 3D interativa mostra a geometria; 2D com risquinhos mostra as ligações; o gráfico mostra os mols antes e depois.

## Referências

- IUPAC. *Compendium of Chemical Terminology* (Gold Book) — verbete "stoichiometric coefficient".
- ATKINS, P.; JONES, L. *Princípios de Química*. Bookman.
- BROWN, T. et al. *Química: a Ciência Central*. Pearson.